Neste exemplo são apresentados a criação e o treinamento de uma rede neural para análise de sentimanto utilizando Transformer. O objetivo é classificar revisões de filmes do IMDb como positivas ou negativas.

Esta implementação não possui um decoder explícito, pois isso não é necessário para a tarefa de classificação de texto (onde a saída é uma classe e não uma sequência de elementos como no problema de tradução automática ou geração de texto).

##**1 - Importando bibliotecas e módulos**

In [5]:
# Importa submódulos específicos do Keras:
# - ops: operações matemáticas e tensoriais (similar ao NumPy)
# - layers: blocos da rede neural (Dense, Conv2D, LSTM, etc.)
# - utils: utilitários gerais (ex: conversão de labels para one-hot)
# - datasets: datasets prontos (ex: MNIST, CIFAR-10)
# - models: classes para montar modelos (Sequential, Model)
from keras import ops, layers, utils, datasets, models

##**2 - Carregando o dataset e separando os dados**

In [6]:
# Define o vocabulário máximo: apenas as 20.000 palavras mais frequentes
# serão consideradas. Palavras raras além desse limite são ignoradas.
max_features = 20000

# Carrega o dataset IMDB (críticas de filmes rotuladas como positivo/negativo)
# num_words=max_features limita o vocabulário às 20k palavras mais comuns
# Retorna dois pares:
# - X_train, Y_train: dados e labels de treino
# - X_val, Y_val: dados e labels de validação
(X_train, Y_train), (X_val, Y_val) = datasets.imdb.load_data(num_words=max_features)

c:\AI\postgrad-ai\.venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [7]:
# Define o comprimento máximo de cada sequência (crítica de filme)
# Sequências serão padronizadas para exatamente 100 "tokens" (palavras)
maxlen = 100

# Padroniza as sequências de treino para o mesmo tamanho (maxlen=100)
# - Sequências maiores que 100: são truncadas
# - Sequências menores que 100: são preenchidas com zeros no início
X_train = utils.pad_sequences(X_train, maxlen=maxlen)

# Mesmo processo aplicado aos dados de validação
X_val = utils.pad_sequences(X_val, maxlen=maxlen)

##**3 - Tratando as Entradas**

In [8]:
# Define a camada de entrada da rede
# shape=(maxlen,) significa que cada amostra é uma sequência de 100 inteiros
# (cada inteiro representa uma palavra)
inputs = layers.Input(shape=(maxlen,))

# Define a dimensão dos embeddings (tamanho do vetor que representa cada palavra)
# Cada palavra será representada por um vetor de 50 números
emb_dims = 50

# Camada de Embedding para as palavras (tokens)
# - input_dim=max_features: tamanho do vocabulário (20.000 palavras)
# - output_dim=emb_dims: cada palavra vira um vetor de 50 dimensões
# Resultado: transforma (batch, 100) → (batch, 100, 50)
token_emb = layers.Embedding(input_dim=max_features, output_dim=emb_dims)(inputs)

# Cria um tensor com as posições [0, 1, 2, ..., 99]
# Representa a posição de cada palavra na sequência
positions = ops.arange(start=0, stop=maxlen, step=1)

# Camada de Embedding para as posições
# - input_dim=maxlen: 100 posições possíveis (0 a 99)
# - output_dim=emb_dims: cada posição vira um vetor de 50 dimensões
# Resultado: transforma (100,) → (100, 50)
pos_emb = layers.Embedding(input_dim=maxlen, output_dim=emb_dims)(positions)

# Soma os embeddings de tokens com os embeddings de posição
# Isso dá à rede informação sobre QUAL palavra é e ONDE ela está na frase
# Técnica fundamental usada em Transformers (Positional Encoding)
x = token_emb + pos_emb

##**4 - Camada de Auto-Atenção (Self-Attention)**

In [9]:
# Define o número de "cabeças" do mecanismo de atenção
# Cada cabeça aprende a prestar atenção em aspectos diferentes da sequência
num_heads = 5

# Camada de Multi-Head Attention
# - num_heads=5: 5 cabeças de atenção paralelas
# - key_dim=emb_dims: dimensão de cada cabeça (50)
# Os dois argumentos (x, x) indicam que query, key e value vêm da mesma fonte
# Isso é chamado de Self-Attention: a sequência presta atenção nela mesma
atencao = layers.MultiHeadAttention(num_heads=num_heads, key_dim=emb_dims)(x, x)

# Dropout: durante o treino, desativa aleatoriamente 10% dos neurônios
# Funciona como uma técnica de regularização para evitar overfitting
atencao_output = layers.Dropout(0.1)(atencao)

# Conexão residual + normalização (padrão Add & Norm dos Transformers)
# - x + atencao_output: soma a entrada original com a saída da atenção
#   Isso é a "conexão residual" — preserva o gradiente e facilita o aprendizado
# - LayerNormalization: normaliza os valores da camada para estabilizar o treino
#   epsilon=1e-6 evita divisão por zero na normalização
saida1 = layers.LayerNormalization(epsilon=1e-6)(x + atencao_output)

##**5 - Camada Feedforward**

In [10]:
# Define o número de neurônios da camada intermediária da FFN
num_neuronios = 32

# --- BLOCO FEED-FORWARD (segundo sub-bloco do Encoder Transformer) ---

# Primeira camada Dense da FFN com ativação ReLU
# Expande a representação: emb_dims(50) → 32 neurônios
# ReLU introduz não-linearidade, permitindo aprender padrões complexos
feed_output = layers.Dense(num_neuronios, activation='relu')(saida1)

# Segunda camada Dense sem ativação
# Projeta de volta para a dimensão original: 32 → emb_dims(50)
# Mantém as dimensões consistentes para a conexão residual a seguir
feed_output = layers.Dense(emb_dims)(feed_output)

# Dropout de 10% para regularização, igual ao bloco de atenção
feed_output = layers.Dropout(0.1)(feed_output)

# Conexão residual + normalização (mesmo padrão Add & Norm do bloco anterior)
# saida1 + feed_output: soma a entrada do bloco com a saída da FFN
x = layers.LayerNormalization(epsilon=1e-6)(saida1 + feed_output)

# --- CABEÇA DE CLASSIFICAÇÃO ---

# Reduz a sequência (batch, 100, 50) → (batch, 50)
# Calcula a média de todos os tokens, criando uma representação global da frase
x = layers.GlobalAveragePooling1D()(x)

# Dropout para regularização antes da classificação
x = layers.Dropout(0.1)(x)

# Camada densa intermediária com ReLU para aprender padrões de classificação
x = layers.Dense(20, activation='relu')(x)

# Mais um Dropout para reduzir overfitting
x = layers.Dropout(0.1)(x)

# Camada de saída com 2 neurônios (positivo / negativo)
# Softmax converte os valores em probabilidades que somam 1.0
# Ex: [0.85, 0.15] → 85% chance de positivo, 15% de negativo
outputs = layers.Dense(2, activation='softmax')(x)

##**6 - Criando e compilando o modelo**

In [11]:
# Instancia o modelo conectando a camada de entrada à camada de saída
# O Keras automaticamente mapeia todas as camadas intermediárias entre elas
model = models.Model(inputs=inputs, outputs=outputs)

# Configura o processo de treinamento:
# - optimizer='adam': algoritmo de otimização adaptativo, ajusta o learning rate
#   automaticamente durante o treino — boa escolha padrão para a maioria dos casos
# - loss='sparse_categorical_crossentropy': função de perda para classificação
#   com múltiplas classes onde os labels são inteiros (0 ou 1, não one-hot)
# - metrics=['accuracy']: monitora a acurácia durante treino e validação
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Treina o modelo com os dados de treino:
# - X_train, Y_train: dados e labels de treino
# - batch_size=32: processa 32 amostras por vez antes de atualizar os pesos
# - epochs=2: passa pelo dataset completo 2 vezes
#   (poucos epochs para teste rápido, mas pode ser insuficiente para convergir)
# - validation_data=(X_val, Y_val): avalia no conjunto de validação ao fim
#   de cada epoch, permitindo monitorar overfitting
model.fit(X_train, Y_train, batch_size=32, epochs=2, validation_data=(X_val, Y_val))

Epoch 1/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 22ms/step - accuracy: 0.8063 - loss: 0.4139 - val_accuracy: 0.8448 - val_loss: 0.3571
Epoch 2/2
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.9049 - loss: 0.2424 - val_accuracy: 0.8430 - val_loss: 0.3667


##**7 - Testando o Modelo**

In [13]:
import re
# Carrega o dicionário de palavras do dataset IMDB
# Mapeia cada palavra ao seu índice numérico — ex: {"the": 1, "film": 2, ...}
indice_da_palavra = datasets.imdb.get_word_index()

def text_to_sequence(text):
  # Remove caracteres especiais e converte o texto para minúsculas e separa em lista de palavras
  palavras = re.sub(r'[^\w\s]', '', text).lower().split()
  sequencia = []

  for palavra in palavras:
    # Verifica se a palavra existe no vocabulário E está dentro do limite
    # de max_features (20.000 palavras mais frequentes)
    if palavra in indice_da_palavra and indice_da_palavra[palavra] < max_features:
      # Adiciona +3 ao índice porque o dataset IMDB reserva os índices 0, 1 e 2
      # para tokens especiais: 0=padding, 1=início de sequência, 2=palavra desconhecida
      sequencia.append(indice_da_palavra[palavra] + 3)

  # Padroniza a sequência para o mesmo tamanho usado no treino (maxlen=100)
  return utils.pad_sequences([sequencia], maxlen=maxlen)

def classify_review(review):
  # Converte o texto da resenha em sequência numérica padronizada
  sequencia = text_to_sequence(review)

  # Roda a sequência pelo modelo e obtém as probabilidades de cada classe
  prediction = model.predict(sequencia)

  # Exibe a probabilidade da classe 1 (positiva)
  print(f' Predição da rede Positiva: {prediction[0][1]:.5f}')
  print(f' Predição da rede Negativa: {prediction[0][0]:.5f}')

  # Classifica como Positiva se a probabilidade da classe positiva (índice 1) for > 0.5
  # caso contrário, classifica como Negativa
  return 'Positiva' if prediction[0][1] > 0.5 else 'Negativa'

# Define a resenha a ser classificada (claramente negativa)
resenha = 'This movie is very bad and boring. I consider it horrible.'

# Chama a função de classificação passando a resenha como argumento
classification = classify_review(resenha)

# Exibe a classificação final
print(f'A resenha é classificada como: {classification}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
 Predição da rede Positiva: 0.00515
 Predição da rede Negativa: 0.99485
A resenha é classificada como: Negativa
